# Lesson 04 - টুল ব্যবহার ডিজাইন প্যাটার্ন

এই পাঠে আপনি মাইক্রোসফ্ট এজেন্ট ফ্রেমওয়ার্ক (পাইথন) ব্যবহার করে AI এজেন্টদের জন্য **টুল ব্যবহার** ডিজাইন প্যাটার্ন শিখবেন। আমরা আলোচনা করব:

- `@tool` ডেকোরেটর এবং টাইপ করা প্যারামিটার সহ ফাংশন টুলগুলি সংজ্ঞায়িত করা
- মডেলকে প্রতিটি টুল কী কাজ করে তা বোঝাতে টুল স্কিমা প্রদান করা
- `approval_mode` দিয়ে টুল কার্যক্রম নিয়ন্ত্রণ করা
- Pydantic মডেল এবং `response_format` মাধ্যমে **গঠনবদ্ধ আউটপুট** ফেরত দেওয়া

পরিস্থিতিটি একটি **ভ্রমণ বুকিং এজেন্ট**, যা গন্তব্য খুঁজে বের করতে পারে, উপলব্ধতা পরীক্ষা করতে পারে এবং ফ্লাইট তথ্য গ্রহণ করতে পারে।


## সেটআপ


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool ডেকোরেটর দিয়ে টুলগুলি সংজ্ঞায়িত করা

`@tool` ডেকোরেটর একটি সাধারণ পাইথন ফাংশনকে এমন একটি টুলে পরিণত করে যা একটি এজেন্ট কল করতে পারে।  
মূল পয়েন্টগুলো:

- **ডকস্ট্রিং** মডেলটি যে টুলটির বিবরণ দেখবে তা হয়ে ওঠে।  
- **টাইপ অ্যানোটেশন** (বর্ণনাসহ `Annotated` সহ) টুলের স্কিমা নির্ধারণ করে।  
- `approval_mode` নিয়ন্ত্রণ করে ব্যবহারকারীকে প্রতিটি কল কার্যকর হওয়ার 전에 অনুমোদন দিতে হবে কিনা।


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## একাধিক টুল সহ একটি এজেন্ট তৈরি করা

সকল তিনটি টুল ক্লায়েন্টকে প্রদান করুন যাতে মডেল প্রয়োজনমতো যেকোনো টুল ব্যবহার করে ব্যবহারকারীর প্রশ্নের উত্তর দিতে পারে।


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## টুল সহ সংগঠিত আউটপুট

`response_format` কে একটি Pydantic মডেলে সেট করার মাধ্যমে, এজেন্টকে মুক্ত ফॉर्म টেক্সটের পরিবর্তে একটি সঠিক টাইপযুক্ত JSON অবজেক্ট ফেরত দিতে বাধ্য করা হয়। এটি তখন উপকারী যখন পরবর্তী কোড ফলাফলকে প্রোগ্রাম্যাটিকালি ব্যবহার করতে হয়।


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## টুল অনুমোদন প্যাটার্ন

`@tool` এ `approval_mode` প্যারামিটার নিয়ন্ত্রণ করে টুল কলগুলি চালানোর আগে মানব অনুমোদন প্রয়োজন কিনা:

| মোড | আচরণ |
|---|---|
| `"never_require"` | টুল স্বয়ংক্রিয়ভাবে চলে — ব্যবহারকারীর নিশ্চিতকরণের দরকার নেই। |
| `"always_require"` | প্রতিটি কল ব্যবহারকারীর অনুমোদিত হওয়ার পরে চালানো হয়। |

যেসব টুলের পার্শ্বপ্রতিক্রিয়া থাকে (যেমন বিমান বুকিং, ক্রেডিট কার্ড চার্জ করা) সেগুলোর জন্য `"always_require"` ব্যবহার করুন যাতে একজন মানুষ সম্পৃক্ত থাকেন।


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## সারাংশ

এই পাঠে আপনি শিখলেন কিভাবে:

1. **টুলস সংজ্ঞায়িত করতে হয়** `@tool` ডেকোরেটর ব্যবহার করে, টাইপ করা প্যারামিটার এবং ডকস্ট্রিংস দিয়ে যা টুল স্কিমা হিসাবে কাজ করে।
2. **একাধিক টুল রচনা করতে হয়** যাতে এজেন্ট সেগুলোকে ক্রমান্বয়ে কল করে জটিল প্রশ্নের উত্তর দিতে পারে।
3. প্যাস করা একটি পিড্যান্টিক মডেল হিসেবে **কাঠামোবদ্ধ আউটপুট ফেরত দিতে হয়** `response_format` হিসাবে।
4. **টুল অনুমোদন নিয়ন্ত্রণ করতে হয়** `approval_mode` দিয়ে যাতে সংবেদনশীল অপারেশনগুলোর জন্য মানুষের নজর রাখা যায়।

এই প্যাটার্নগুলো নির্ভরযোগ্য, প্রোডাকশন-রেডি এজেন্ট তৈরি করার জন্য ভিত্তি রচনা করে যা নিরাপদে বাহ্যিক সিস্টেমের সাথে ইন্টারঅ্যাক্ট করতে পারে।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকারোক্তি**:  
এই নথিটি এআই অনুবাদ সেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। আমরা সঠিকতার জন্য সচেষ্ট থাকি, কিন্তু স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথি তার নিজস্ব ভাষায় কর্তৃত্বমূলক উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে কোনো ভুল বুঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
